In [1]:
import csv
import pandas as pd
from pathlib import Path


def _read_evolving_csv(csv_path: Path) -> pd.DataFrame:
    """
    Liest experiment_summary.csv auch dann fehlerfrei ein, wenn neuere Runs
    zusätzliche Spalten (z.B. mean_c_at_best_loss, mean_c_last_epoch) haben als ältere Runs.
    """
    try:
        return pd.read_csv(csv_path)
    except pd.errors.ParserError:
        with open(csv_path, "r") as f:
            rows = list(csv.reader(f))
        header = rows[0]
        max_cols = max(len(r) for r in rows)
        idx = header.index("fold_c_indices") + 1
        if max_cols > len(header):
            extra_cols = ["mean_c_at_best_loss", "std_c_at_best_loss", "mean_c_last_epoch", "std_c_last_epoch"]
            header = header[:idx] + extra_cols[:max_cols - len(header)] + header[idx:]

        padded_rows = [
            r[:idx] + [None] * (max_cols - len(r)) + r[idx:] if len(r) < max_cols else r
            for r in rows[1:]
        ]
        df = pd.DataFrame(padded_rows, columns=header)
        for col in df.columns:
            if col.startswith(("mean_", "std_", "lr", "dropout", "loss_alpha", "weight_decay")):
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df


# Ordner (relativ zu diesem Notebook) -> Anzeigename in der Tabelle
MODEL_DIRS = {
    "ABMIL":    "ABMIL (wsi only)",
    "TRANSMIL": "TransMIL(wsi only)",
    "MCAT":     "MCAT (wsi + genomic)",
    "mcat_bidirectional": "MCAT bidirektional (wsi + genomic, eigenes Modell)",
    "MOTCAT":   "MOTCAT (wsi + genomic)",
    "SURVPATH": "SurvPath (wsi + genomic)",
    "healnet":  "HEALNet (wsi + genomic)",
    "SNN":      "SNN (genomic only, 4917 Pathway-Gene)",
    "SNN_20k":  "SNN (genomic only, 20530 Xena-Gene)",
    "MLP":      "MLP (genomic only, 4917 Pathway-Gene)",
    "MLP_20k":  "MLP (genomic only, 20530 Xena-Gene)",
    "SNN_mcat": "SNN (genomic only, 2854 MCAT/MOTCAT-Gene)",
    "MLP_mcat": "MLP (genomic only, 2854 MCAT/MOTCAT-Gene)",
}
DATASETS = ["KIRC", "GBMLGG", "KIRP"]


def create_last_epoch_table(
    epochs: int,
    base_dir: Path = Path("../models_training_notebooks"),
    model_dirs: dict = MODEL_DIRS,
    datasets: list = DATASETS,
    decimals: int = 3,
) -> pd.DataFrame:
    """
    Fasst pro Modell-Ordner den jeweils neuesten Run mit `epochs` Epochen je Kohorte zusammen und zeigt
    nur den C-Index @ Last Epoch (mean ± std über die Folds) pro Dataset sowie Overall.
    Overall = Mittel der Dataset-Means ± Mittel der Dataset-Stds (nur wenn alle Datasets vorliegen).
    """
    fmt = lambda m, s: f"{m:.{decimals}f} ± {s:.{decimals}f}"
    rows = []
    for folder, name in model_dirs.items():
        row = {"Model": name, **{d: "–" for d in datasets}, "Overall": "–"}
        csv_path = Path(base_dir) / folder / "runs" / "experiment_summary.csv"
        if not csv_path.exists():
            rows.append(row)
            continue

        df = _read_evolving_csv(csv_path)
        df = df[pd.to_numeric(df["epochs"], errors="coerce") == epochs].copy()
        # Neuere Runs (SNN, MLP, SurvPath, HEALNet) speichern nur mean_c_index, waehlen aber immer die letzte Epoche
        # (model_selection == "last_epoch") -> dann ist mean_c_index der C-Index @ Last Epoch.
        if "model_selection" in df.columns:
            last = df["model_selection"] == "last_epoch"
            for stat in ("mean", "std"):
                col = f"{stat}_c_last_epoch"
                if col not in df.columns:
                    df[col] = float("nan")
                df.loc[last, col] = df.loc[last, col].fillna(df.loc[last, f"{stat}_c_index"])
        if "mean_c_last_epoch" not in df.columns:
            rows.append(row)
            continue

        # Neuesten Run pro Kohorte anhand des Zeitstempels im run_name (..._YYYYMMDD_HHMMSS)
        df["_ts"] = df["run_name"].str.extract(r"(\d{8}_\d{6})$")[0]
        df = df.sort_values("_ts").drop_duplicates(subset="cohort", keep="last")
        df["Dataset"] = df["cohort"].str.replace("TCGA-", "", regex=False)
        df = df.dropna(subset=["mean_c_last_epoch"]).set_index("Dataset")

        means, stds = [], []
        for d in datasets:
            if d in df.index:
                m, s = float(df.loc[d, "mean_c_last_epoch"]), float(df.loc[d, "std_c_last_epoch"])
                row[d] = fmt(m, s)
                means.append(m)
                stds.append(s)
        if len(means) == len(datasets):
            row["Overall"] = fmt(sum(means) / len(means), sum(stds) / len(stds))
        rows.append(row)

    return pd.DataFrame(rows)


def show_table(table: pd.DataFrame, title: str, datasets: list = DATASETS):
    """Zeigt die Tabelle an; der beste Mittelwert je Spalte (Kohorten und Overall) ist fett (bei Gleichstand alle besten)."""
    def bold_best(col: pd.Series) -> list:
        means = pd.to_numeric(col.str.split(" ± ").str[0], errors="coerce")  # "–" -> NaN
        return ["font-weight: bold" if m == means.max() else "" for m in means]

    styler = (
        table.style.hide(axis="index")
        .apply(bold_best, subset=datasets + ["Overall"], axis=0)
        .set_caption(title)
        .set_table_styles([{"selector": "caption", "props": "font-weight: bold; text-align: left; padding: 6px 0;"}])
    )
    display(styler)


table_20 = create_last_epoch_table(epochs=20)
show_table(table_20, "C-Index @ Last Epoch – 20 Epochen")

Model,KIRC,GBMLGG,KIRP,Overall
ABMIL (wsi only),0.699 ± 0.094,0.744 ± 0.076,0.754 ± 0.099,0.732 ± 0.090
TransMIL(wsi only),0.700 ± 0.050,0.748 ± 0.062,0.699 ± 0.090,0.716 ± 0.067
MCAT (wsi + genomic),0.738 ± 0.054,0.809 ± 0.046,0.792 ± 0.154,0.780 ± 0.085
MOTCAT (wsi + genomic),0.735 ± 0.043,0.808 ± 0.057,0.791 ± 0.144,0.778 ± 0.082
SurvPath (wsi + genomic),0.696 ± 0.037,0.817 ± 0.051,0.773 ± 0.134,0.762 ± 0.074
HEALNet (wsi + genomic),0.703 ± 0.047,0.830 ± 0.030,0.764 ± 0.165,0.766 ± 0.081
"SNN (genomic only, 4917 Pathway-Gene)",0.684 ± 0.076,0.822 ± 0.036,0.812 ± 0.130,0.773 ± 0.081
"SNN (genomic only, 20530 Xena-Gene)",0.706 ± 0.065,0.820 ± 0.036,0.768 ± 0.162,0.765 ± 0.088
"MLP (genomic only, 4917 Pathway-Gene)",0.706 ± 0.061,0.829 ± 0.041,0.785 ± 0.151,0.773 ± 0.084
"MLP (genomic only, 20530 Xena-Gene)",0.697 ± 0.064,0.829 ± 0.035,0.767 ± 0.151,0.764 ± 0.084


In [2]:
# SNN_mcat / MLP_mcat wurden nur mit 20 Epochen trainiert
table_10 = create_last_epoch_table(
    epochs=10, model_dirs={k: v for k, v in MODEL_DIRS.items() if not k.endswith("_mcat")})
show_table(table_10, "C-Index @ Last Epoch – 10 Epochen")

Model,KIRC,GBMLGG,KIRP,Overall
ABMIL (wsi only),0.706 ± 0.092,0.757 ± 0.072,0.739 ± 0.112,0.734 ± 0.092
TransMIL(wsi only),0.701 ± 0.040,0.761 ± 0.062,0.770 ± 0.110,0.744 ± 0.070
MCAT (wsi + genomic),0.719 ± 0.030,0.813 ± 0.046,0.806 ± 0.139,0.780 ± 0.072
MOTCAT (wsi + genomic),0.718 ± 0.045,0.822 ± 0.047,0.803 ± 0.145,0.781 ± 0.079
SurvPath (wsi + genomic),0.674 ± 0.041,0.817 ± 0.054,0.767 ± 0.094,0.753 ± 0.063
HEALNet (wsi + genomic),0.721 ± 0.042,0.833 ± 0.037,0.780 ± 0.160,0.778 ± 0.080
"SNN (genomic only, 4917 Pathway-Gene)",0.685 ± 0.068,0.823 ± 0.037,0.822 ± 0.116,0.777 ± 0.074
"SNN (genomic only, 20530 Xena-Gene)",0.689 ± 0.067,0.822 ± 0.039,0.791 ± 0.148,0.767 ± 0.085
"MLP (genomic only, 4917 Pathway-Gene)",0.683 ± 0.060,0.830 ± 0.043,0.795 ± 0.132,0.769 ± 0.079
"MLP (genomic only, 20530 Xena-Gene)",0.689 ± 0.065,0.831 ± 0.037,0.781 ± 0.136,0.767 ± 0.079
